# Comparing Efficient Multi-Head Attention Implementations

This code notebook compares different ways to implement causal multi-head attention used in decoder-style LLMs like GPT, Llama, etc.

In [2]:
import torch

torch.manual_seed(123)
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device =  torch.device("cpu")

print(f"Using device: {device}")
print(f"Pytorch Version: {torch.__version__}")

batch_size = 8
context_len  = 1024
embed_dim = 768
embeddings = torch.randn((batch_size, context_len, embed_dim), device=device)

Using device: cuda
Pytorch Version: 2.5.1+cu121


- To run all the code in this notebook, please ensure you update to at least PyTorch 2.5 (FlexAttention is not included in earlier PyTorch releases)
- If the code cell above shows a PyTorch version lower than 2.5, you can upgrade your PyTorch installation by uncommenting and running the following code cell (Please note that PyTorch 2.5 requires Python 3.9 or later)
- For more specific instructions and CUDA versions, please refer to the official installation guide at https://pytorch.org

<br>
&nbsp;

## 1. CausalAttention MHA wrapper 

     Menial implementation of Multi-Head Attention

In [3]:
import torch.nn as nn


class CasualAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # register buffer is used to move the tensors to the cuda device; not the default feature
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))


    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights  = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values 
        return context_vec
    

class MHA_Wrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CasualAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

        self.out_proj = nn.Linear(d_out * num_heads, d_out * num_heads)

    def forward(self, x):
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)
    
mha = MHA_Wrapper(
    d_in=embed_dim,
    d_out=embed_dim//12,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha(embeddings)
print(out.shape)
print(out)

torch.Size([8, 1024, 768])
tensor([[[ 0.0042, -0.2265,  0.3464,  ...,  0.2234,  0.3797,  0.1758],
         [ 0.0979, -0.0153,  0.1987,  ...,  0.4223,  0.1606, -0.0673],
         [ 0.1590, -0.2563, -0.0018,  ...,  0.2600,  0.1795,  0.1389],
         ...,
         [ 0.0302, -0.0450, -0.0284,  ...,  0.0383, -0.0404,  0.0329],
         [ 0.0310, -0.0368, -0.0173,  ...,  0.0447, -0.0321,  0.0333],
         [ 0.0281, -0.0384, -0.0201,  ...,  0.0427, -0.0331,  0.0450]],

        [[ 0.1418,  0.0939, -0.2882,  ..., -0.4492, -0.0861, -0.8380],
         [ 0.3817, -0.1967,  0.0999,  ..., -0.0505,  0.0487, -0.1510],
         [-0.1646, -0.2212,  0.1931,  ...,  0.0157,  0.0193, -0.2100],
         ...,
         [ 0.0147, -0.0484, -0.0015,  ...,  0.0430, -0.0424,  0.0301],
         [ 0.0132, -0.0430, -0.0021,  ...,  0.0335, -0.0328,  0.0341],
         [ 0.0196, -0.0353, -0.0096,  ...,  0.0220, -0.0468,  0.0263]],

        [[-0.1502,  0.0623, -0.1925,  ..., -0.0016,  0.0088,  0.0810],
         [ 0.0408,

<br>
&nbsp;

## 2. The multi-head attention class from chapter 3

     Above implementation has a bottleneck it does sequential calculation while this does it in parallel and effeciently use the compute device

In [18]:
class MHA(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
            b, num_tokens, d_in = x.shape
    
            keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
            queries = self.W_query(x)
            values = self.W_value(x)
    
            # We implicitly split the matrix by adding a `num_heads` dimension
            # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
            keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
            values = values.view(b, num_tokens, self.num_heads, self.head_dim)
            queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
    
            # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
            keys = keys.transpose(1, 2)
            queries = queries.transpose(1, 2)
            values = values.transpose(1, 2)
    
            # Compute scaled dot-product attention (aka self-attention) with a causal mask
            attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head
    
            # Original mask truncated to the number of tokens and converted to boolean
            mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
    
            # Use the mask to fill attention scores
            attn_scores.masked_fill_(mask_bool, -torch.inf)
    
            attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
            attn_weights = self.dropout(attn_weights)
    
            # Shape: (b, num_tokens, num_heads, head_dim)
            context_vec = (attn_weights @ values).transpose(1, 2)
    
            # Combine heads, where self.d_out = self.num_heads * self.head_dim
            context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
            context_vec = self.out_proj(context_vec)  # optional projection
    
            return context_vec

mha_parallel = MHA(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_parallel(embeddings)
print(out.shape)

torch.Size([8, 1024, 768])


<br>
&nbsp;

## 3. An alternative multi-head attention with combined weights

- The code for the `MultiHeadAttentionCombinedQKV` class below is based on code that was kindly shared by [Rayed Bin Wahed](https://github.com/rasbt/LLMs-from-scratch/discussions/51)
- The main difference between the `MultiHeadAttentionCombinedQKV` class and the `MultiHeadAttention` class used in chapter 3 is that `MultiHeadAttentionCombinedQKV` uses a single weight matrix, `self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)` instead of separate weight matrices:

  - `self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)`
  - `self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)`
  - `self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)`

- Here, `self.qkv` combines all three weight matrices `self.W_query`, `self.W_key`, and `self.W_value` to carry out the query, key, and value computation in a single step
- Using `q, k, v = qkv.unbind(0)`, we obtain the individual query, key, and value tensors, which are then used similarly to the query, key, and value tensors in the `MultiHeadAttention` class in chapter 3

In [22]:
class MHA_CombinedQKV(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
            super().__init__()
    
            assert d_out % num_heads == 0, "d_out is indivisible by num_heads"
    
            self.num_heads = num_heads
            self.context_length = context_length
            self.head_dim = d_out // num_heads
    
            self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
            self.proj = nn.Linear(d_out, d_out)
            self.dropout = nn.Dropout(dropout)
    
            self.register_buffer(
                "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
            )

    def forward(self, x):
        batch_size , num_tokens, embed_dim = x.shape

        qkv = self.qkv(x)

        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        qkv = qkv.permute(2,0,3,1,4)

        queries, keys, values = qkv.unbind(0)

        attn_scores = queries @ keys.transpose(-2, -1)
        attn_scores = attn_scores.masked_fill_(
             self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        context_vec = context_vec.transpose(1, 2)
        context_vec = context_vec.contiguous().view(batch_size, num_tokens, embed_dim)

        context_vec = self.proj(context_vec)

        return context_vec

mha_combined_qkv = MHA_CombinedQKV(

    d_in=embed_dim,
    d_out= embed_dim,
    context_length=context_len,
    dropout=0.0,
    num_heads=12,
    qkv_bias=False
).to(device)

out = mha_combined_qkv(embeddings)
print(out.shape)

torch.Size([8, 1024, 768])
